# 03 — Pemberian keterangan Gambar (JoyCaption)

Notebook lanjutan `identity-lora-pipeline`: membaca hasil crop tubuh penuh dari data `data/cropped/<identitas>` (modul 001) dan hasil crop wajah dari data `data/face/<identitas>` (modul 002), memberikan keterangan pada gambar dengan model `JoyCaption`, menghasilkan teks keterangan gambar ke `data/cropped/<identitas>` dan `data/face/<identitas>`.

## 1. Import dan logger

In [ ]:
import logging
import sys
from dataclasses import dataclass
from itertools import islice
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

# Contoh: 2026-09-26 10:30:00,123 | INFO     | image_captioning | ...
LOG_FORMAT = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"


def setup_logger(name: str) -> logging.Logger:
    """Membuat logger dengan format teks.

    Args:
        name: Nama logger.

    Returns:
        Logger yang siap dipakai.
    """
    created = logging.getLogger(name)
    if created.handlers:
        return created

    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(LOG_FORMAT))
    created.addHandler(handler)
    created.setLevel(logging.INFO)
    return created


logger = setup_logger("image_captioning")

## 2. Konfigurasi

In [ ]:
# Kernel notebook biasanya berjalan dari folder notebooks/, jadi path relatif "data/raw"
# tidak ketemu. Root project dicari dengan naik dari folder kerja sampai ada data/raw/.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "raw").is_dir()),
    Path.cwd(),
)
CROPPED_DIR = PROJECT_ROOT / "data" / "cropped"
FACE_DIR = PROJECT_ROOT / "data" / "face"

MODEL_NAME = "fancyfeast/llama-joycaption-beta-one-hf-llava"
TEMP = 0.7
PROMPT = """Write a straightforward caption for this image. Structure the description in this exact order:

1. Camera angle and framing (e.g. close-up, medium shot, full body, front-facing, side angle, from above/below)

2. Face: describe ONLY expression and gaze direction (e.g. smiling, neutral, looking at camera, looking away, looking down). Do NOT describe eye shape, eye color, nose shape, lip shape, jawline, or any permanent facial structure.

3. Clothing fit: describe how the clothing fits on the body (loose, fitted, tight, oversized, tucked in, etc.). Do NOT describe body build, height, shoulder width, or body proportions — these should not be captioned.

4. Clothing details: describe garment type, color, pattern, material texture, and any accessories in detail.

Be specific and use clear, concrete language. Do not skip any of the four sections. Do not mention skin tone, facial structure, or body shape anywhere in the caption.

Output format:
- Write the caption as a single continuous paragraph, not a numbered or bulleted list.
- Do NOT include section labels, numbers, or headers (e.g. no "1.", "Face:", "Clothing:") in the output.
- Separate each described element with a comma, not a period, so the four sections read as one flowing sentence.
- Do not wrap the caption in quotation marks.
- Do not add any preamble, explanation, or commentary before or after the caption (e.g. no "Here is the caption:").
- Output only the caption text itself, nothing else.
- Keep the total caption under 60 words."""

TRIGGER_WORD = "<nama>"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
CAPTION_IMAGE_LIMIT = 5
CAPTION_BATCH_SIZE = 2

## 3. Error pipeline

In [ ]:
# Definisi error ada di notebook terpisah; %run mengeksekusinya di namespace kernel ini.
%run "{PROJECT_ROOT / 'notebooks' / '00_pipeline_errors.ipynb'}"

## 4. Pemberian Keterangan

In [ ]:
@dataclass
class ImageCaption:
    caption: str
    prompt: str
    temperature: float


def load_vlm_model(
    model_name: str = MODEL_NAME,
) -> tuple[AutoProcessor, LlavaForConditionalGeneration]:
    """Memuat processor dan model JoyCaption."""
    processor = AutoProcessor.from_pretrained(model_name)
    model = LlavaForConditionalGeneration.from_pretrained(
        model_name, torch_dtype=torch.bfloat16, device_map="auto"
    )
    return processor, model


def _load_images(image_paths: list[Path]) -> list[Image.Image]:
    """Membaca dan mengonversi gambar ke RGB.

    Raises:
        ImageReadError: Salah satu gambar gagal dibaca atau rusak.
    """
    images = []
    for image_path in image_paths:
        try:
            with Image.open(image_path) as image:
                images.append(image.convert("RGB"))
        except OSError as e:
            raise ImageReadError(f"Gambar gagal dibaca: {image_path}") from e
    return images


def generate_caption(
    image_path: Path,
    processor: AutoProcessor,
    model: LlavaForConditionalGeneration,
    prompt: str = PROMPT,
    temperature: float = TEMP,
) -> ImageCaption:
    """Menghasilkan caption untuk satu gambar melalui inferensi batch ukuran satu."""
    return generate_captions(
        [image_path], processor, model, prompt, temperature
    )[0]


def generate_captions(
    image_paths: list[Path],
    processor: AutoProcessor,
    model: LlavaForConditionalGeneration,
    prompt: str = PROMPT,
    temperature: float = TEMP,
) -> list[ImageCaption]:
    """Menghasilkan caption untuk beberapa gambar dalam satu inferensi batch.

    Raises:
        ImageReadError: Salah satu gambar di batch gagal dibaca.
        CaptionGenerationError: Model gagal menghasilkan output untuk batch ini.
    """
    if not image_paths:
        return []

    images = _load_images(image_paths)

    # Format percakapan multimodal resmi transformers: "content" berupa list
    # blok {"type": "image"}/{"type": "text"}, bukan string biasa, supaya
    # placeholder token gambar ikut disisipkan oleh apply_chat_template dan
    # jumlahnya cocok dengan gambar yang dikirim ke processor.
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    conversation_text = processor.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[conversation_text] * len(images),
        images=images,
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    try:
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=temperature,
        )
    except RuntimeError as e:
        raise CaptionGenerationError(
            f"Model captioning gagal pada batch {len(images)} gambar"
        ) from e
    input_token_count = inputs["input_ids"].shape[1]
    captions = processor.tokenizer.batch_decode(
        output[:, input_token_count:], skip_special_tokens=True
    )
    return [
        ImageCaption(caption=caption.strip(), prompt=prompt, temperature=temperature)
        for caption in captions
    ]


def save_caption(image_path: Path, caption: ImageCaption) -> Path:
    """Menyimpan caption di samping gambar dengan nama dasar yang sama."""
    caption_path = image_path.with_suffix(".txt")
    caption_path.write_text(caption.caption + "\n", encoding="utf-8")
    return caption_path


def post_process_caption(caption_path: Path, trigger_word: str = TRIGGER_WORD) -> Path:
    """Menambahkan trigger word di awal caption secara idempoten."""
    caption = caption_path.read_text(encoding="utf-8").strip()
    trigger_word = trigger_word.strip()
    if not trigger_word:
        return caption_path

    prefix = f"{trigger_word},"
    if not caption.lower().startswith(prefix.lower()):
        caption = f"{trigger_word}, {caption}" if caption else trigger_word
    caption_path.write_text(caption + "\n", encoding="utf-8")
    return caption_path


def list_image_paths(input_dir: Path, limit: int) -> list[Path]:
    """Mengambil maksimal `limit` gambar dari semua folder identitas."""
    image_paths = (
        image_path
        for identity_dir in sorted(input_dir.iterdir())
        if identity_dir.is_dir()
        for image_path in sorted(identity_dir.iterdir())
        if image_path.suffix.lower() in IMAGE_EXTENSIONS
    )
    return list(islice(image_paths, limit))


def caption_images(
    image_paths: list[Path],
    processor: AutoProcessor,
    model: LlavaForConditionalGeneration,
    prompt: str = PROMPT,
    temperature: float = TEMP,
    trigger_word: str = TRIGGER_WORD,
    batch_size: int = CAPTION_BATCH_SIZE,
) -> list[Path]:
    """Memproses gambar per batch, menyimpan dan mem-post-process tiap caption.

    Batch yang gagal dibaca atau gagal di-generate dilewati dan dicatat ke log,
    batch lain tetap diproses (granularitas per batch karena inferensi dilakukan
    sekaligus per batch, bukan per gambar).
    """
    if batch_size < 1:
        raise ValueError("batch_size harus lebih besar dari 0")

    caption_paths = []
    for batch_start in range(0, len(image_paths), batch_size):
        image_batch = image_paths[batch_start : batch_start + batch_size]
        try:
            captions = generate_captions(
                image_batch, processor, model, prompt, temperature
            )
        except ImageReadError as e:
            logger.warning("Batch dilewati, gambar gagal dibaca: %s", e)
            continue
        except CaptionGenerationError as e:
            logger.warning("Batch dilewati, model captioning gagal: %s", e)
            continue
        for image_path, caption in zip(image_batch, captions):
            caption_path = save_caption(image_path, caption)
            post_process_caption(caption_path, trigger_word)
            caption_paths.append(caption_path)
            logger.info("Caption tersimpan: %s", caption_path)
    return caption_paths


def run_captioning(
    cropped_limit: int = CAPTION_IMAGE_LIMIT,
    face_limit: int = CAPTION_IMAGE_LIMIT,
    trigger_word: str = TRIGGER_WORD,
    batch_size: int = CAPTION_BATCH_SIZE,
) -> dict[str, list[Path]]:
    """Menjalankan captioning batch untuk cropped dan face."""
    processor, model = load_vlm_model()
    cropped_paths = list_image_paths(CROPPED_DIR, cropped_limit)
    face_paths = list_image_paths(FACE_DIR, face_limit)
    return {
        "cropped": caption_images(
            cropped_paths, processor, model, trigger_word=trigger_word, batch_size=batch_size
        ),
        "face": caption_images(
            face_paths, processor, model, trigger_word=trigger_word, batch_size=batch_size
        ),
    }

In [ ]:
# Runner "define -> demonstrate": menghasilkan caption maksimal CAPTION_IMAGE_LIMIT
# gambar dari data/cropped/ dalam batch CAPTION_BATCH_SIZE dan menyimpan setiap hasil
# sebagai file .txt di samping gambar. Sel ini membutuhkan bobot JoyCaption dan GPU;
# sengaja tidak dieksekusi otomatis saat pemeriksaan kode (lihat README.md).
_runner_cropped_processor, _runner_cropped_model = load_vlm_model()
_runner_cropped_paths = list_image_paths(CROPPED_DIR, CAPTION_IMAGE_LIMIT)
_runner_cropped_caption_paths = caption_images(
    _runner_cropped_paths,
    _runner_cropped_processor,
    _runner_cropped_model,
    batch_size=CAPTION_BATCH_SIZE,
)
print(f"{len(_runner_cropped_caption_paths)} caption cropped tersimpan")

In [ ]:
# Runner "define -> demonstrate": menghasilkan caption maksimal CAPTION_IMAGE_LIMIT
# gambar dari data/face/ dalam batch CAPTION_BATCH_SIZE dan menyimpan setiap hasil
# sebagai file .txt di samping gambar. Sel ini membutuhkan bobot JoyCaption dan GPU;
# sengaja tidak dieksekusi otomatis saat pemeriksaan kode (lihat README.md).
_runner_face_processor, _runner_face_model = load_vlm_model()
_runner_face_paths = list_image_paths(FACE_DIR, CAPTION_IMAGE_LIMIT)
_runner_face_caption_paths = caption_images(
    _runner_face_paths,
    _runner_face_processor,
    _runner_face_model,
    batch_size=CAPTION_BATCH_SIZE,
)
print(f"{len(_runner_face_caption_paths)} caption face tersimpan")

In [ ]:
# Orkestrasi captioning batch: memproses maksimal CAPTION_IMAGE_LIMIT gambar cropped
# dan maksimal CAPTION_IMAGE_LIMIT gambar face dengan batch CAPTION_BATCH_SIZE.
# Hasil disimpan sebagai file .txt dengan stem yang sama seperti nama gambar.
captioning_results = run_captioning(
    cropped_limit=CAPTION_IMAGE_LIMIT,
    face_limit=CAPTION_IMAGE_LIMIT,
    batch_size=CAPTION_BATCH_SIZE,
)
print(
    f"Caption tersimpan: {len(captioning_results['cropped'])} cropped, "
    f"{len(captioning_results['face'])} face"
)